In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================
# Config
# ============================================================
BASE_DIR = Path("./")

INPUT_GFEVD = BASE_DIR / "gfevd_all.npy"
INPUT_DATE = BASE_DIR / "tvpvar_input_scaled.csv"
INPUT_EVENTS = BASE_DIR / "events_window0.csv"

OUT_DIR = BASE_DIR / "event_persistence_outputs"
OUT_DIR.mkdir(exist_ok=True)

H = 20  # persistence를 볼 horizon
USE_ABS_AUC = True

# ============================================================
# 1. Load
# ============================================================
gfevd = np.load(INPUT_GFEVD)   # shape: (T, k, k)

df_date = pd.read_csv(INPUT_DATE)
df_date["Date"] = pd.to_datetime(df_date["Date"])

df_events = pd.read_csv(INPUT_EVENTS)
df_events["Date"] = pd.to_datetime(df_events["Date"])

T, k, _ = gfevd.shape
dates = pd.to_datetime(df_date["Date"].values[-T:])

print(f"[INFO] T={T}, k={k}")
print(f"[INFO] Number of events: {len(df_events)}")

# ============================================================
# 2. Delta total 계산
# ============================================================
delta = np.abs(gfevd[1:] - gfevd[:-1])   # shape: (T-1, k, k)
offdiag_mask = ~np.eye(k, dtype=bool)

delta_total = delta[:, offdiag_mask].reshape(T - 1, -1).sum(axis=1)
delta_dates = dates[1:]

df_delta = pd.DataFrame({
    "Date": delta_dates,
    "DeltaTotal": delta_total
})

date_to_idx = {pd.Timestamp(d): i for i, d in enumerate(delta_dates)}

# ============================================================
# 3. Persistence feature 계산 함수
# ============================================================
def compute_persistence_features(series_values, event_idx, horizon):
    """
    series_values: 1D np.array of DeltaTotal
    event_idx: event가 발생한 index (delta series 기준)
    horizon: 이후 몇 시점까지 볼지

    baseline은 event 직전 값으로 둠
    path_h = series[event_idx + h] - series[event_idx - 1]
    """
    if event_idx == 0:
        return None

    max_h = min(horizon, len(series_values) - 1 - event_idx)
    if max_h < 0:
        return None

    baseline = series_values[event_idx - 1]
    path = np.array([
        series_values[event_idx + h] - baseline
        for h in range(max_h + 1)
    ], dtype=float)

    abs_path = np.abs(path)

    peak = abs_path.max()
    peak_h = abs_path.argmax()

    initial = abs_path[0]

    # Half-life
    if initial == 0:
        half_life = 0
    else:
        half_threshold = 0.5 * initial
        half_candidates = np.where(abs_path <= half_threshold)[0]
        half_life = int(half_candidates[0]) if len(half_candidates) > 0 else np.nan

    # DurationAboveHalf
    if initial == 0:
        duration_above_half = 0
    else:
        duration_above_half = int(np.sum(abs_path >= 0.5 * initial))

    # AUC
    if USE_ABS_AUC:
        auc = abs_path.sum()
    else:
        auc = path.sum()

    return {
        "Baseline": baseline,
        "InitialResponse": path[0],
        "Peak": peak,
        "PeakHorizon": int(peak_h),
        "HalfLife": half_life,
        "DurationAboveHalf": duration_above_half,
        "AUC": auc,
        "MaxHorizonUsed": int(max_h),
        "Path": path
    }

# ============================================================
# 4. Event별 계산
# ============================================================
summary_rows = []
path_rows = []

for event_date in df_events["Date"]:
    event_date = pd.Timestamp(event_date)

    if event_date not in date_to_idx:
        print(f"[WARN] Event date {event_date.date()} not found in delta_dates, skipped")
        continue

    event_idx = date_to_idx[event_date]
    result = compute_persistence_features(delta_total, event_idx, H)

    if result is None:
        print(f"[WARN] Event date {event_date.date()} could not compute persistence, skipped")
        continue

    summary_rows.append({
        "Date": event_date,
        "Baseline": result["Baseline"],
        "InitialResponse": result["InitialResponse"],
        "Peak": result["Peak"],
        "PeakHorizon": result["PeakHorizon"],
        "HalfLife": result["HalfLife"],
        "DurationAboveHalf": result["DurationAboveHalf"],
        "AUC": result["AUC"],
        "MaxHorizonUsed": result["MaxHorizonUsed"]
    })

    for h, val in enumerate(result["Path"]):
        path_rows.append({
            "Date": event_date,
            "Horizon": h,
            "Response": val
        })

# ============================================================
# 5. Save
# ============================================================
df_summary = pd.DataFrame(summary_rows).sort_values("Date").reset_index(drop=True)
df_path = pd.DataFrame(path_rows).sort_values(["Date", "Horizon"]).reset_index(drop=True)

summary_path = OUT_DIR / "event_persistence_window0.csv"
path_path = OUT_DIR / "event_persistence_window0_path_long.csv"

df_summary.to_csv(summary_path, index=False)
df_path.to_csv(path_path, index=False)

print(f"[INFO] Saved summary: {summary_path}")
print(f"[INFO] Saved path long: {path_path}")
print("[INFO] Done")

[INFO] T=1035, k=7
[INFO] Number of events: 52
[INFO] Saved summary: event_persistence_outputs\event_persistence_window0.csv
[INFO] Saved path long: event_persistence_outputs\event_persistence_window0_path_long.csv
[INFO] Done
